In [1]:
from drive_service.auth_service import load_creds
from drive_service.drive_client import get_drive_service

creds = load_creds()

drive = get_drive_service(creds)


In [2]:
from drive_service.logging_utils import get_logger


logger = get_logger()

In [3]:
import os
from scan_directory.cli import _get_root_name
from pathlib import Path
root = "1hxSSsFQNMo63L_dDEbyNtVTa1I7IUjQC"
root_prefix = _get_root_name(drive, root)

scan_output =  Path( root_prefix)

os.makedirs(scan_output, exist_ok=True)
root_prefix


'ACETO'

In [4]:
from drive_service.drive_client import list_children


sub_folders = list_children(drive, root)
print( "Sotto Cartelle", len(sub_folders))
sub_folders

Sotto Cartelle 24


[{'id': '1c6bxLwo4oh0N4Hh0KvNQSaAsBvaW7pnq',
  'name': 'CANUTI Cristina',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1T0V1rj44Lx4ZYYCm8efaLUIFNHW1FpmZ',
  'name': 'BRUCIAPORCI GIUSEPPINA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '18K3UExA9cfLIRNSO3lxJEyNvRS1OBYoa',
  'name': 'BRIZZI EMANUELA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1cPpG2Z8wC-fxP99gpg52TA0wD5AYDVMp',
  'name': 'BURCHIELLI FEDERICA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1Mn8kWMxWeQlsSPUYHLVH2dUA0roy6jXC',
  'name': 'FABBRI FEDERICO',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1dIuDkI0Q-SdXiwNNlrmGxg8eTPiaEa7f',
  'name': 'CANESCHI GIULIA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1pEONrbedR53luQVryNaD2eigPwVN79xP',
  'name': 'ACETO CATERINA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1ddirq0Rgy3QS9LfMOjLFhwLlsz7X8oJZ',
  'name': 'GAETANI SARA',
  'mimeType': 'application/vnd

In [21]:
from concurrent.futures import ThreadPoolExecutor, as_completed

from scan_directory.config import exclude_terms_normalized
from scan_directory.scan_service import build_folder_report

workers= 8 

reports = []
total_included = 0
with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [
            pool.submit(
                build_folder_report,
                creds,
                emp,
                exclude_terms_normalized,
                root_prefix=root_prefix,
            )
            for emp in sub_folders
        ]
        for i, f in enumerate(as_completed(futures), 1):
            report = f.result()
            reports.append(report)
            total_included += report["counts"]["included"]
            logger.info(
                "Progress %s/%s employees, %s files",
                i,
                len(futures),
                total_included,
            )

KeyboardInterrupt: 

In [ ]:
filtered = [file  for r in reports for file in r["filtered"]]
next( f for f in filtered if f["type"] != "folder" )

{'employee': 'ACETO CATERINA',
 'file_id': '1SGMJo4M-pG_v3GMSc2vcvWWf2TjxjZZp',
 'file_name': 'presenze 2023.zip',
 'drive_path': '/ACETO/ACETO CATERINA/presenze/presenze 2023.zip',
 'type': 'file',
 'reason': 'zip_no_pdf_members'}

In [ ]:
from drive_service.schema import IndexFile


included_map: dict[str, IndexFile] = {}
filtered_map: dict[str, IndexFile] = {}

for report in reports:
    for item in report["included"]:
        file_id = item.get("file_id")
        if not file_id:
            continue
        if file_id in included_map:
            logger.warning("Duplicate file_id in included map: %s (last one wins)", file_id)
        included_map[file_id] = IndexFile(**item)
    for item in report["filtered"]:
        file_id = item.get("file_id")
        if not file_id:
            continue
        if file_id in filtered_map:
            logger.warning("Duplicate file_id in filtered map: %s (last one wins)", file_id)
        filtered_map[file_id] = IndexFile(**item)

Adding to filtered map: {'employee': 'FOROTTI ALESSANDRO', 'employee_id': '13Pxew40QLP5DftDoddGgiQibG2_rGiyE', 'file_id': '13XEE4tL6DsjOaw_DjQS6Jxe8fhKflUIr', 'file_name': 'BUSTE PAGA', 'drive_path': '/ACETO/FOROTTI ALESSANDRO/BUSTE PAGA', 'type': 'folder', 'reason': 'buste paga'}
Adding to filtered map: {'employee': 'FABBRI FEDERICO', 'employee_id': '1Mn8kWMxWeQlsSPUYHLVH2dUA0roy6jXC', 'file_id': '1vj30cvATQYw2MesKW0dMcJouwAqFaXgy', 'file_name': 'BUSTE PAGA', 'drive_path': '/ACETO/FABBRI FEDERICO/BUSTE PAGA', 'type': 'folder', 'reason': 'buste paga'}
Adding to filtered map: {'employee': 'BELARDI ROBERTA', 'employee_id': '13oyxtOuAKhVocEv9r_Ko7A-VyxcoEuMD', 'file_id': '1oDRFRWib9MOBsMTGvyRpI9acbCTYw3Im', 'file_name': 'BUSTE PAGA', 'drive_path': '/ACETO/BELARDI ROBERTA/BUSTE PAGA', 'type': 'folder', 'reason': 'buste paga'}
Adding to filtered map: {'employee': 'BURCHIELLI FEDERICA', 'employee_id': '1cPpG2Z8wC-fxP99gpg52TA0wD5AYDVMp', 'file_id': '1BFE109mZ06Xmoa_QT6iA_FkBHpYS1gx6', 'file_

In [8]:
from drive_service.index.map_index import MapIndex

included_path =  scan_output / "included_index.json"
filtered_path =  scan_output / "filtered_index.json" 
included_index = MapIndex.generate_index(root, len(sub_folders), included_map)
filtered_index = MapIndex.generate_index(root, len(sub_folders), filtered_map)
included_index.save_index(included_path)
filtered_index.save_index(filtered_path)